## 2.Inspect

In [64]:
import pandas as pd

YOUR_ID = 3246

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=YOUR_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1155, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           57
building_type         0
ceiling_height_m    139
price_kzt             0
dtype: int64
                   count unique         top freq        mean         std  min  \
listing_id          1155   1080   ALM-10932    3         NaN         NaN  NaN   
listed_date         1155    247  2026-04-21   10         NaN         NaN  NaN   
district            1155     48       Medeu  148         NaN         NaN  NaN   
rooms               1155      6           2  431         NaN      

In [65]:
print("rooms уникальные:", df["rooms"].unique())
print()
print("area_m2 примеры:", df["area_m2"].unique()[:20])
print()
print("price_kzt примеры:", df["price_kzt"].unique()[:20])
print()
print("district уникальные (все 48):")
print(sorted(df["district"].unique()))

rooms уникальные: ['4' '2' '3' '1' '5' 'studio']

area_m2 примеры: ['90.8' '52.1' '69.8' '82.8' '56.2' '64.3' '116.1' '56.9' '48.6' '36.6'
 '53.5' '67.6' '133.0' '59.0' '48.2' '56.3' '67.2' '110.2' '57.0' '35.4']

price_kzt примеры: ['41,560,000' '27700000' '78490000' '95690000' '30040000' '59000000'
 '63730000' '25720000' '26120000' '29170000' '29050000' '43390000'
 '177470000' '70470000' '38740000' '29 060 000 ₸' '27670000' '60140000'
 '25760000' '23580000']

district уникальные (все 48):
[' Alatau ', ' Almaly ', ' Auezov ', ' Bostandyk ', ' Medeu ', ' Nauryzbay ', ' Turksib ', ' Zhetysu ', 'ALATAU', 'ALMALY', 'AUEZOV', 'Alatau', 'Alatauskiy', 'Almalinskiy', 'Almaly', 'Auezov', 'Auezovskiy', 'BOSTANDYK', 'Bostandyk', 'Bostandykskiy', 'MEDEU', 'Medeu', 'Medeuskiy', 'NAURYZBAY', 'Nauryzbay', 'Nauryzbayskiy', 'TURKSIB', 'Turksib', 'Turksibskiy', 'ZHETYSU', 'Zhetysu', 'Zhetysuskiy', 'alatau', 'almaly', 'auezov', 'bostandyk', 'medeu', 'nauryzbay', 'turksib', 'zhetysu', 'Алатауский', 'Алма

## Data Inspection Findings

- `rooms`, `area_m2`, `price_kzt` are stored as `object` instead of numeric types.
  - `rooms` contains the string `"studio"` alongside numeric strings.
  - `price_kzt` has three formats: plain digits, comma-separated, and space-separated with a `₸` symbol.
- `listing_id`: 1155 rows but only 1080 unique values → duplicates/reposts present.
- `floor` has a minimum of -1 (impossible value, likely a sentinel).
- `year_built` has a minimum of 0 (impossible, likely a sentinel) and 57 missing values.
- `ceiling_height_m` has 139 missing values.
- `district` has 48 raw string variants (casing, whitespace, transliteration suffix `-skiy`, Russian names) that map down to 8 canonical districts: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu.

In [66]:
validation_spec = {
    "listing_id":       {"dtype": "string", "notes": "unique per real listing, but duplicates/reposts exist"},
    "listed_date":      {"dtype": "datetime"},
    "district":         {"dtype": "category", "allowed": ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]},
    "rooms":            {"dtype": "int", "min": 0, "max": 10, "notes": "'studio' -> 0"},
    "area_m2":          {"dtype": "float", "min": 10, "max": 500},
    "floor":            {"dtype": "int", "min": 1, "max": 60, "notes": "current min=-1 is impossible"},
    "total_floors":     {"dtype": "int", "min": 1, "max": 60},
    "year_built":       {"dtype": "int", "min": 1950, "max": 2026, "notes": "current min=0 is impossible/sentinel"},
    "building_type":    {"dtype": "category"},
    "ceiling_height_m": {"dtype": "float", "min": 2.0, "max": 4.5},
    "price_kzt":        {"dtype": "float", "min": 1_000_000, "max": 500_000_000},
}
validation_spec

{'listing_id': {'dtype': 'string',
  'notes': 'unique per real listing, but duplicates/reposts exist'},
 'listed_date': {'dtype': 'datetime'},
 'district': {'dtype': 'category',
  'allowed': ['Alatau',
   'Almaly',
   'Auezov',
   'Bostandyk',
   'Medeu',
   'Nauryzbay',
   'Turksib',
   'Zhetysu']},
 'rooms': {'dtype': 'int', 'min': 0, 'max': 10, 'notes': "'studio' -> 0"},
 'area_m2': {'dtype': 'float', 'min': 10, 'max': 500},
 'floor': {'dtype': 'int',
  'min': 1,
  'max': 60,
  'notes': 'current min=-1 is impossible'},
 'total_floors': {'dtype': 'int', 'min': 1, 'max': 60},
 'year_built': {'dtype': 'int',
  'min': 1950,
  'max': 2026,
  'notes': 'current min=0 is impossible/sentinel'},
 'building_type': {'dtype': 'category'},
 'ceiling_height_m': {'dtype': 'float', 'min': 2.0, 'max': 4.5},
 'price_kzt': {'dtype': 'float', 'min': 1000000, 'max': 500000000}}

## 3.Clean

In [67]:
dup_ids = df["listing_id"][df["listing_id"].duplicated(keep=False)]
dup_rows = df[df["listing_id"].isin(dup_ids)].sort_values("listing_id")
print(dup_rows.shape)
dup_rows.head(20)

(147, 11)


,listing_id,listed_date,district,rooms,area_m2,floor,total_floors,year_built,building_type,ceiling_height_m,price_kzt
8,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
962,ALM-10003,2026-05-04,Auezov,2,48.6,2,5,1978.0,panel,2.55,28320000
819,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
935,ALM-10009,2025-12-03,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,42700000
179,ALM-10009,2026-01-19,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,39400000
453,ALM-10016,2026-01-29,Auezov,1,39.7,13,25,2022.0,monolith,2.80,32100000
477,ALM-10016,2026-01-20,Auezov,1,39.7,13,25,2022.0,monolith,2.80,35130000
255,ALM-10021,2026-04-06,Alatau,3,71.3,9,12,1988.0,panel,2.69,32300000
609,ALM-10021,2026-05-09,Alatau,3,71.3,9,12,1988.0,panel,2.69,30090000
52,ALM-10056,2026-02-08,Bostandyk,4,107.2,9,9,2006.0,monolith,2.81,101060000


In [68]:
before = len(df)

df["listed_date"] = pd.to_datetime(df["listed_date"])

df_clean = (
    df.sort_values("listed_date")
      .drop_duplicates(subset="listing_id", keep="last")
      .reset_index(drop=True)
)

after = len(df_clean)
print(f"Rows before: {before}, after dedup: {after}, removed: {before - after}")

Rows before: 1155, after dedup: 1080, removed: 75


In [69]:
canonical_districts = ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]

ru_map = {
    "Алатауский": "Alatau",
    "Алмалинский": "Almaly",
    "Ауэзовский": "Auezov",
    "Бостандыкский": "Bostandyk",
    "Медеуский": "Medeu",
    "Наурызбайский": "Nauryzbay",
    "Турксибский": "Turksib",
    "Жетысуский": "Zhetysu",
}

def clean_district(x):
    x = x.strip()
    if x in ru_map:
        return ru_map[x]
    x = x.lower()
    for suffix in ["inskiy", "skiy"]:
        if x.endswith(suffix):
            x = x[: -len(suffix)]
            break
    for d in canonical_districts:
        if d.lower().startswith(x):
            return d
    return None

df_clean["district"] = df_clean["district"].apply(clean_district)

print(df_clean["district"].unique())
print(df_clean["district"].isna().sum())

['Almaly' 'Zhetysu' 'Alatau' 'Medeu' 'Bostandyk' 'Auezov' 'Turksib'
 'Nauryzbay']
0


Districts normalized to 8 canonical values by stripping whitespace,lowercasing,mapping Russian names via a dictionary and stripping transliteration suffixes (`-skiy`/`-inskiy`). `Almalinskiy` required a `startswith` match rather than exact match due to an irregular suffix pattern

In [70]:
# rooms: 'studio' в 0,остальное в число
df_clean["rooms"] = df_clean["rooms"].replace("studio", "0").astype(int)

# area_m2: убрать "m²",меняем запятую на точку,приведем к float
df_clean["area_m2"] = (
    df_clean["area_m2"]
    .astype(str)
    .str.replace("m²", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

# price_kzt: убираем запятые,пробелы,символ ₸
df_clean["price_kzt"] = (
    df_clean["price_kzt"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₸", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
    .astype(float)
)

print(df_clean[["rooms", "area_m2", "price_kzt"]].dtypes)
print(df_clean[["rooms", "area_m2", "price_kzt"]].describe())

rooms          int64
area_m2      float64
price_kzt    float64
dtype: object
             rooms      area_m2     price_kzt
count  1080.000000  1080.000000  1.080000e+03
mean      2.236111    68.907963  4.631151e+07
std       1.051409    59.780981  2.521701e+07
min       0.000000    25.300000  1.144000e+07
25%       1.000000    45.000000  2.828750e+07
50%       2.000000    58.750000  4.019000e+07
75%       3.000000    78.650000  5.732500e+07
max       5.000000   879.000000  1.930000e+08


## 4.Sentinels & outliers

In [71]:
print("floor value counts (низкие и высокие):")
print(df_clean["floor"].value_counts().sort_index().head(5))
print()
print("year_built value counts (низкие):")
print(df_clean["year_built"].value_counts().sort_index().head(10))
print()
print("ceiling_height_m value counts (низкие):")
print(df_clean["ceiling_height_m"].value_counts().sort_index().head(10))

floor value counts (низкие и высокие):
floor
-1     11
 1    141
 2    124
 3    152
 4    109
Name: count, dtype: int64

year_built value counts (низкие):
year_built
0.0       20
1950.0     3
1951.0     2
1952.0     6
1953.0     5
1954.0     3
1955.0     3
1956.0     2
1957.0     4
1958.0     6
Name: count, dtype: int64

ceiling_height_m value counts (низкие):
ceiling_height_m
0.00    21
2.50     7
2.51    24
2.52    19
2.53    26
2.54    11
2.55    21
2.56    14
2.57    14
2.58    15
Name: count, dtype: int64


In [72]:
import numpy as np

sentinel_counts = {
    "floor": (df_clean["floor"] == -1).sum(),
    "year_built": (df_clean["year_built"] == 0).sum(),
    "ceiling_height_m": (df_clean["ceiling_height_m"] == 0).sum(),
}
print("Sentinels found:", sentinel_counts)

df_clean.loc[df_clean["floor"] == -1, "floor"] = np.nan
df_clean.loc[df_clean["year_built"] == 0, "year_built"] = np.nan
df_clean.loc[df_clean["ceiling_height_m"] == 0, "ceiling_height_m"] = np.nan

print()
print("Missing values after sentinel replacement:")
print(df_clean.isna().sum())

Sentinels found: {'floor': np.int64(11), 'year_built': np.int64(20), 'ceiling_height_m': np.int64(21)}

Missing values after sentinel replacement:
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                11
total_floors          0
year_built           74
building_type         0
ceiling_height_m    151
price_kzt             0
dtype: int64


In [73]:
for col, spec in validation_spec.items():
    if "min" in spec and "max" in spec and col in df_clean.columns:
        bad = df_clean[(df_clean[col] < spec["min"]) | (df_clean[col] > spec["max"])]
        if len(bad) > 0:
            print(f"{col}: {len(bad)} rows outside [{spec['min']}, {spec['max']}]")
            print(bad[[col]].head())
            print()

area_m2: 8 rows outside [10, 500]
     area_m2
1      736.0
163    604.0
214    647.0
257    530.0
504    668.0



In [74]:
big_area = df_clean[df_clean["area_m2"] > 500]
big_area["price_per_m2"] = big_area["price_kzt"] / big_area["area_m2"]
print(big_area[["listing_id", "rooms", "area_m2", "price_kzt", "price_per_m2", "district"]])

     listing_id  rooms  area_m2   price_kzt   price_per_m2 district
1     ALM-10910      3    736.0  54050000.0   73437.500000  Zhetysu
163   ALM-10276      2    604.0  33630000.0   55678.807947  Turksib
214   ALM-11122      2    647.0  40670000.0   62859.350850  Zhetysu
257   ALM-10072      2    530.0  64270000.0  121264.150943    Medeu
504   ALM-10921      3    668.0  38560000.0   57724.550898  Zhetysu
763   ALM-10909      2    565.0  32610000.0   57716.814159   Alatau
944   ALM-10440      3    879.0  48370000.0   55028.441411   Auezov
1044  ALM-10659      2    544.0  25030000.0   46011.029412  Turksib


C:\Users\shack\AppData\Local\Temp\ipykernel_18856\2943618679.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  big_area["price_per_m2"] = big_area["price_kzt"] / big_area["area_m2"]


In [ ]:
big_area["area_div10"] = big_area["area_m2"] / 10
big_area["price_per_m2_fixed"] = big_area["price_kzt"] / big_area["area_div10"]
print(big_area[["area_m2", "area_div10", "price_per_m2_fixed"]])

      area_m2  area_div10  price_per_m2_fixed
1       736.0        73.6        7.343750e+05
163     604.0        60.4        5.567881e+05
214     647.0        64.7        6.285935e+05
257     530.0        53.0        1.212642e+06
504     668.0        66.8        5.772455e+05
763     565.0        56.5        5.771681e+05
944     879.0        87.9        5.502844e+05
1044    544.0        54.4        4.601103e+05


C:\Users\shack\AppData\Local\Temp\ipykernel_18856\3734711820.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  big_area["area_div10"] = big_area["area_m2"] / 10
C:\Users\shack\AppData\Local\Temp\ipykernel_18856\3734711820.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  big_area["price_per_m2_fixed"] = big_area["price_kzt"] / big_area["area_div10"]


In [76]:
mask = df_clean["area_m2"] > 500
print(f"Fixing {mask.sum()} rows with area_m2 > 500 (dividing by 10)")
df_clean.loc[mask, "area_m2"] = df_clean.loc[mask, "area_m2"] / 10

print(df_clean["area_m2"].describe())

Fixing 8 rows with area_m2 > 500 (dividing by 10)
count    1080.000000
mean       64.597130
std        31.520127
min        25.300000
25%        45.000000
50%        58.550000
75%        78.325000
max       495.000000
Name: area_m2, dtype: float64


In [77]:
bad_floor = df_clean[df_clean["floor"] > df_clean["total_floors"]]
print(f"Rows where floor > total_floors: {len(bad_floor)}")
print(bad_floor[["listing_id", "floor", "total_floors"]])

Rows where floor > total_floors: 9
     listing_id  floor  total_floors
10    ALM-10729   15.0            12
125   ALM-10851   13.0            12
151   ALM-10944    4.0             3
313   ALM-10044   30.0            25
616   ALM-10636    6.0             5
697   ALM-10628    6.0             5
832   ALM-10266   13.0            12
992   ALM-10074   11.0             9
1008  ALM-10695   14.0            12


In [78]:
mask = df_clean["floor"] > df_clean["total_floors"]
print(f"Setting floor to NaN for {mask.sum()} rows where floor > total_floors")
df_clean.loc[mask, "floor"] = np.nan

print(df_clean["floor"].isna().sum())

Setting floor to NaN for 9 rows where floor > total_floors
20


In [79]:
def iqr_outliers(group, col):
    q1 = group[col].quantile(0.25)
    q3 = group[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return group[(group[col] < lower) | (group[col] > upper)]

outliers_list = []
for district, group in df_clean.groupby("district"):
    out = iqr_outliers(group, "price_kzt")
    outliers_list.append(out)

price_outliers = pd.concat(outliers_list)
print(f"Total price outliers (grouped by district): {len(price_outliers)}")
print(price_outliers[["listing_id", "district", "price_kzt"]].sort_values("district"))

Total price outliers (grouped by district): 36
     listing_id   district    price_kzt
356   ALM-10963     Alatau   67380000.0
444   ALM-10392     Alatau   60140000.0
512   ALM-10095     Alatau   58750000.0
582   ALM-10661     Alatau   50900000.0
827   ALM-11028     Alatau   72750000.0
993   ALM-10345     Alatau   56300000.0
1046  ALM-10061     Almaly  146480000.0
451   ALM-10439     Almaly  125650000.0
219   ALM-11128     Almaly  148590000.0
148   ALM-10885     Almaly  118740000.0
141   ALM-10328     Almaly  125100000.0
300   ALM-10612     Auezov   89990000.0
779   ALM-10193     Auezov   87470000.0
828   ALM-10842     Auezov   87440000.0
894   ALM-10529     Auezov   87540000.0
964   ALM-11003  Bostandyk  129570000.0
885   ALM-10713  Bostandyk  162880000.0
798   ALM-11006  Bostandyk  167190000.0
789   ALM-10403  Bostandyk  193000000.0
363   ALM-10567  Bostandyk  119960000.0
173   ALM-10534      Medeu  159220000.0
731   ALM-10783      Medeu  149210000.0
737   ALM-11098      Medeu  14922

In [80]:
df_clean["is_price_outlier"] = df_clean["listing_id"].isin(price_outliers["listing_id"])
print(df_clean["is_price_outlier"].sum(), "rows flagged as price outliers")

36 rows flagged as price outliers


## Sentinels, Impossible Values, and Outliers — Decisions

- **Sentinels** (`floor == -1`, `year_built == 0`, `ceiling_height_m == 0`): converted to `NaN`.These are clear placeholder values (gap in distribution before real data starts),not genuine measurements.Handled by imputation in the missing-data step rather than dropped,to preserve the rest of each row.
- **`area_m2` > 500** (8 rows): corrected by dividing by 10.Verified via `price_per_m2` — the "raw" price-per-m² was 5-10x below the dataset norm and became normal after the /10 correction,confirming a data-entry error (extra digit) rather than genuine outsized apartments.
- **`floor` > `total_floors`** (9 rows): set `floor` to `NaN`.Cannot determine which of the two values is wrong,so the unreliable one is treated as missing rather than guessed at or the row dropped.
- **`price_kzt` outliers via IQR, grouped by `district`** (36 rows): kept as-is,flagged with `is_price_outlier`.All outliers are on the high side and plausible (premium listings within each district),not data errors — dropping or altering them would remove real market signal.

In [81]:
import os
os.makedirs("data", exist_ok=True)
df_clean.to_csv("data/almaty_apartments_clean.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (1080, 12)
